In [1]:
import pandas as pd
import numpy as np
df = pd.read_stata('A1_kommune.dta')
df

,nr,kommune,taxrev,taxrate,pop
0,101,Københavns Kommune,44170.335938,23.799999,528208
1,147,Frederiksberg Kommune,6682.439941,23.100000,96718
2,151,Ballerup Kommune,4598.704102,25.500000,47652
3,153,Brøndby Kommune,3121.204834,24.500000,33795
4,155,Dragør Kommune,858.017944,24.799999,13564
...,...,...,...,...,...
93,840,Rebild Kommune,1957.348511,25.100000,28852
94,846,Mariagerfjord Kommune,3186.054932,25.700001,42604
95,849,Jammerbugt Kommune,3241.256592,25.299999,38927
96,851,Aalborg Kommune,16330.091797,25.400000,197426


Problem 1

In [2]:
descriptive_analysis = df[['taxrev', 'taxrate', 'pop']].describe()
descriptive_analysis.round(2)


,taxrev,taxrate,pop
count,98.00,98.00,98.00
mean,4477.34,25.21,56475.89
std,5251.18,0.91,62925.30
min,211.23,22.80,1969.00
25%,2466.70,24.80,29997.75
50%,3317.85,25.30,43475.00
75%,4786.06,25.70,59733.00
max,44170.34,27.80,528208.00


Problem 2.3

In [3]:
# #ESTIMATION model (1)

x = df['taxrate'].values # alle 98 x_i-værdier på én gang (som et array)
y = np.log(df['taxrev']).values # alle 98 y_i-værdier (log-transformerede)

x_bar = np.mean(x) #gennemsnittet af taxrates
y_bar = np.mean(y) #gennemsnittet af taxrevs

n =len(x)

delta1_hat = (np.sum((y-y_bar)*(x-x_bar)))/(np.sum((x-x_bar)**2)) #Estimation af delta1_hat
delta0_hat = y_bar - delta1_hat * x_bar #Estimation af delta0_hat

y_hat = delta0_hat + delta1_hat * x #Den estimerede model
residualer = y - y_hat 


SST = np.sum((y-y_bar)**2) #Total sum of squares
SSR = np.sum(residualer ** 2) #Regression sum of squares
SSE = np.sum((y_hat - y_bar)**2) #Sum of squared errors
sigma2_hat = SSR/(n-2) #Residualvariansen

se_delta1 = np.sqrt(sigma2_hat / np.sum((x-x_bar)**2)) #Standardfejlen for delta1_hat (sigma2_hat/SST_x)
se_delta0 = np.sqrt(sigma2_hat * (1/n + x_bar**2 / np.sum((x - x_bar)**2)))


R2 = SSE/SST

print(f"delta1_hat = {delta1_hat}")
print(f"delta0_hat = {delta0_hat}")
print(f"Standardfejlen for delta1_hat = {se_delta1}")
print(f"Standardfejlen for delta0_hat = {se_delta0}")







delta1_hat = -0.1426168829202652
delta0_hat = 11.698205947875977
Standardfejlen for delta1_hat = 0.08495860546827316
Standardfejlen for delta0_hat = 2.1430251598358154


Problem 2.5

In [4]:
#ESTIMATION model (2)

def mlr_OLS(X, y):
    n = X.shape[0] #antal rækker = 98
    k = X.shape[1] - 1  # antal forklarende variable (kolonner) (ikke inkl. konstant)

    beta_hat = np.linalg.inv(X.T @ X) @ X.T @ y

    y_hat = X @ beta_hat
    residualer = y - y_hat

    y_bar = np.mean(y)
    SST = np.sum((y - y_bar) ** 2)          # Total sum of squares
    SSR = np.sum(residualer ** 2)           # Sum of squared residuals
    SSE = np.sum((y_hat - y_bar) ** 2)      # Explained sum of squares

    sigma2_hat = SSR / (n - k - 1)          # Residualvariansen
    R2 = SSE / SST

    var_beta_hat = sigma2_hat * np.linalg.inv(X.T @ X)
    se_beta = np.sqrt(np.diag(var_beta_hat))    # Standardfejlen for hver beta_hat

    for i, beta in enumerate(beta_hat):
        print(f"beta{i}_hat = {beta}")
        print(f"Standardfejlen for beta{i}_hat = {se_beta[i]}")

    print(f"R2 = {R2}")

    return beta_hat, se_beta, R2

df['const'] = 1
df['logpop'] = np.log(df['pop'])

X_25 = df[['const', 'taxrate', 'logpop']].values
y_25 = np.log(df['taxrev']).values

beta_hat, se_beta, R2 = mlr_OLS(X_25, y_25)

beta0_hat = -2.8021654924070325
Standardfejlen for beta0_hat = 0.37557591397810947
beta1_hat = 0.022622264058431964
Standardfejlen for beta1_hat = 0.012453946193824094
beta2_hat = 0.9711010669897469
Standardfejlen for beta2_hat = 0.014392938188572449
R2 = 0.9801408344014539


Problem 3.3

In [5]:
#Simple regression of taxrate on log(pop)

import statsmodels.api as sm

X = np.log(df[['pop']]).copy() #udtrækker taxrate-kolonnen (forklarende variabel) og laver en kopi, som kaldes X
X['const'] = 1 #tilføjer en konstant til X
y = df['taxrate'] #den afhængige variabel

model = sm.OLS(y, X)
results = model.fit()
print(results.summary())

df['res1'] = results.resid

                            OLS Regression Results                            
Dep. Variable:                taxrate   R-squared:                       0.039
Model:                            OLS   Adj. R-squared:                  0.029
Method:                 Least Squares   F-statistic:                     3.862
Date:                Thu, 17 Sep 2026   Prob (F-statistic):             0.0523
Time:                        20:49:19   Log-Likelihood:                -127.16
No. Observations:                  98   AIC:                             258.3
Df Residuals:                      96   BIC:                             263.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
pop           -0.2273      0.116     -1.965      0.0

In [6]:
#Simple regression of log(taxrev) on res1

y = np.log(df['taxrev'])
X = df[['res1']].copy()
X['const'] = 1

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 taxrev   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.010
Method:                 Least Squares   F-statistic:                   0.06626
Date:                Thu, 17 Sep 2026   Prob (F-statistic):              0.797
Time:                        20:49:19   Log-Likelihood:                -112.50
No. Observations:                  98   AIC:                             229.0
Df Residuals:                      96   BIC:                             234.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
res1           0.0226      0.088      0.257      0.7